## Extracting data from Materials Project to feed NanoMECommons Projects in Granta MI database
* Author: David Mercier / Date: 2024-10
* This script is used to extract data from the Materials Project database (https://next-gen.materialsproject.org/)
* The dataflow is the option 1 in the schema below.

<img src="..\doc\pictures\dataflow1.jpg" height="250">
<img src="..\doc\pictures\dataflow2.jpg" height="250">

In [ ]:
######################################################
### Importing the necessary libraries
#from mp_api.client import MPRester
import pymatgen as mp
import pymatgen.core as mg
from pymatgen.ext.matproj import MPRester
from pymatgen.io.cif import CifParser
from matminer.featurizers.structure import JarvisCFID
import pandas as pd
import os
from mp_api.client import MPRester as MPRester_API
from pymatgen.analysis.elasticity import ElasticTensor
from pymatgen.analysis.elasticity import ElasticTensorExpansion

######################################################
### Input the material ID (QUERY)
# Configure PMG_MAPI_KEY in ~/.pmgrc.yaml before connecting.
QUERY = 'mp-13' # Iron
mpr = MPRester()
mpr_api = MPRester_API()

# Set some parameters
SourceRef = 'The Materials Project'
MethodRef = 'Atomistic simulations (DFT, MD...)'
TRef = 0

# Import data using pymatgen

In [189]:
structures = mpr.get_structures('Ti')
print(structures)

Retrieving MaterialsDoc documents:   0%|          | 0/10 [00:00<?, ?it/s]

[Structure Summary
Lattice
    abc : 12.217542213801519 11.626678546347817 12.310470847386489
 angles : 89.32063507040851 93.28746052606118 87.76716177889094
 volume : 1744.3182239748066
      A : 12.21015799 0.23761671 -0.35201975
      B : 0.22898197 11.62422855 0.06731935
      C : -0.35320933 0.0816896 12.30513155
    pbc : True True True
PeriodicSite: Ti (1.92, 1.371, 9.001) [0.1764, 0.1092, 0.7359]
PeriodicSite: Ti (8.197, 8.729, 11.77) [0.6858, 0.7301, 0.9725]
PeriodicSite: Ti (3.684, 7.572, 2.424) [0.2955, 0.644, 0.2019]
PeriodicSite: Ti (6.065, 5.571, 2.814) [0.4949, 0.4674, 0.2403]
PeriodicSite: Ti (5.333, 2.189, 8.645) [0.4542, 0.174, 0.7146]
PeriodicSite: Ti (6.42, 8.246, 2.129) [0.518, 0.6975, 0.184]
PeriodicSite: Ti (4.685, 5.198, 9.773) [0.3988, 0.4334, 0.8032]
PeriodicSite: Ti (1.948, 6.254, 9.683) [0.1724, 0.529, 0.789]
PeriodicSite: Ti (11.79, 8.471, 10.06) [0.9767, 0.7029, 0.8412]
PeriodicSite: Ti (8.504, 9.861, 2.666) [0.6876, 0.8326, 0.2318]
PeriodicSite: Ti (2.0, 

In [190]:
# Collecting doc from the database
docs = mpr.summary.search(material_ids=[QUERY])
#print(docs)
doc = docs[0]
chemical_info = doc.chemsys
symmetry_info = doc.symmetry
CSVal = symmetry_info.crystal_system
CSVal2 = symmetry_info.point_group
print(CSVal)
if CSVal == 'Cubic':
    CSVal = 'Cubic'

C:\Users\dmercier\AppData\Local\Temp\ipykernel_58088\1364678435.py:2: DeprecationWarning: Accessing summary data through MPRester.summary is deprecated. Please use MPRester.materials.summary instead.
  docs = mpr.summary.search(material_ids=[QUERY])


Retrieving SummaryDoc documents:   0%|          | 0/1 [00:00<?, ?it/s]

Cubic


In [191]:
list_of_available_fields = mpr.summary.available_fields
print(list_of_available_fields)

['builder_meta', 'nsites', 'elements', 'nelements', 'composition', 'composition_reduced', 'formula_pretty', 'formula_anonymous', 'chemsys', 'volume', 'density', 'density_atomic', 'symmetry', 'property_name', 'material_id', 'deprecated', 'deprecation_reasons', 'last_updated', 'origins', 'warnings', 'structure', 'task_ids', 'uncorrected_energy_per_atom', 'energy_per_atom', 'formation_energy_per_atom', 'energy_above_hull', 'is_stable', 'equilibrium_reaction_energy_per_atom', 'decomposes_to', 'xas', 'grain_boundaries', 'band_gap', 'cbm', 'vbm', 'efermi', 'is_gap_direct', 'is_metal', 'es_source_calc_id', 'bandstructure', 'dos', 'dos_energy_up', 'dos_energy_down', 'is_magnetic', 'ordering', 'total_magnetization', 'total_magnetization_normalized_vol', 'total_magnetization_normalized_formula_units', 'num_magnetic_sites', 'num_unique_magnetic_sites', 'types_of_magnetic_species', 'bulk_modulus', 'shear_modulus', 'universal_anisotropy', 'homogeneous_poisson', 'e_total', 'e_ionic', 'e_electronic',

C:\Users\dmercier\AppData\Local\Temp\ipykernel_58088\4093616438.py:1: DeprecationWarning: Accessing summary data through MPRester.summary is deprecated. Please use MPRester.materials.summary instead.
  list_of_available_fields = mpr.summary.available_fields


In [192]:
#Getting materials data using fields
Material_Data = mpr.materials.summary.search(material_ids=[QUERY], fields = ['bulk_modulus', 'density', 'shear_modulus', 'homogeneous_poisson', 'e_ij_max', 'grain_boundaries','structure', 'k_vrh'])
print(Material_Data)

Retrieving SummaryDoc documents:   0%|          | 0/1 [00:00<?, ?it/s]

[MPDataDoc<SummaryDoc>(
density=7.902858230744394,
structure=Structure Summary
Lattice
    abc : 2.477813028804854 2.477809133849768 4.054295207189474
 angles : 90.00003845025343 90.00031950636809 109.469836683298
 volume : 23.468168469356993
      A : 2.33614509 0.00011167 -0.82582293
      B : -1.16807798 2.02304724 -0.8260822
      C : 1.17007387 2.027305 3.31032796
    pbc : True True True
PeriodicSite: Fe (1.169, 2.025, 0.8292) [0.5, 0.5, 0.5]
PeriodicSite: Fe (1.168, 2.023, -1.652) [1.0, 1.0, 1e-08],
grain_boundaries=[GBSearchData(sigma=3, type='tilt', gb_energy=0.4231747636833746, rotation_angle=109.47122063449069), GBSearchData(sigma=5, type='tilt', gb_energy=1.8920508128903335, rotation_angle=36.86989764584402), GBSearchData(sigma=5, type='tilt', gb_energy=1.5602073830337888, rotation_angle=53.13010235415599), GBSearchData(sigma=3, type='tilt', gb_energy=0.5083194482566669, rotation_angle=180.0), GBSearchData(sigma=9, type='tilt', gb_energy=1.7539170879038837, rotation_angle=3

In [193]:
DensityVal = [Mdata.density for Mdata in Material_Data]
LattParamAVal = [Mdata.structure.lattice.a for Mdata in Material_Data] #in Angstrom
LattParamBVal = [Mdata.structure.lattice.b for Mdata in Material_Data] #in Angstrom
LattParamCVal = [Mdata.structure.lattice.c for Mdata in Material_Data] #in Angstrom
caRatio_valueVal = LattParamCVal[0]/LattParamAVal[0]
BulkModulusVal = [Mdata.bulk_modulus for Mdata in Material_Data] #in GPa
BulkModulusVal = BulkModulusVal[0] #in GPa
BulkModulusVoigtVal = BulkModulusVal['voigt'] #in GPa
BulkModulusReussVal = BulkModulusVal['reuss'] #in GPa
BulkModulusVRHVal = BulkModulusVal['vrh'] #in GPa
ShearModulusVal = [Mdata.shear_modulus for Mdata in Material_Data] #in GPa
ShearModulusVal = ShearModulusVal[0] #in GPa
ShearModulusVoigtVal = ShearModulusVal['voigt'] #in GPa
ShearModulusReussVal = ShearModulusVal['reuss'] #in GPa
ShearModulusVRHVal = ShearModulusVal['vrh'] #in GPa
PoissonRatioVal = [Mdata.homogeneous_poisson for Mdata in Material_Data]
print(PoissonRatioVal)

[0.353]


In [194]:
strainTensor = [[0 for _ in range(3)] for _ in range(3)]
strainTensor[2][2] = 1  # Set the last position to 1
print(strainTensor)

with mpr_api as mpr:
                qresults = mpr.elasticity.search(material_ids=[QUERY])
                elastic_tensor = ElasticTensor.from_voigt(qresults[0].elastic_tensor.ieee_format)
                #ecs = ElasticTensorExpansion.get_effective_ecs(strainTensor)

#print(elastic_tensor)

[[0, 0, 0], [0, 0, 0], [0, 0, 1]]


C:\Users\dmercier\AppData\Local\Temp\ipykernel_58088\2849619306.py:6: DeprecationWarning: Accessing elasticity data through MPRester.elasticity is deprecated. Please use MPRester.materials.elasticity instead.
  qresults = mpr.elasticity.search(material_ids=[QUERY])


Retrieving ElasticityDoc documents:   0%|          | 0/1 [00:00<?, ?it/s]

In [195]:
if CSVal == 'Cubic':
    C11Val = elastic_tensor[0,0,0,0]
    C12Val = elastic_tensor[0,0,1,1]
    C44Val = elastic_tensor[1,0,0,1]
    C_names = ['C11', 'C12', 'C44']
    C_values = [C11Val, C12Val, C44Val]
elif CSVal == 'Hexagonal':
    C11Val = elastic_tensor[0,0,0,0]
    C12Val = elastic_tensor[0,0,1,1]
    C13Val = elastic_tensor[0,0,2,2]
    C33Val = elastic_tensor[2,0,0,2]
    C44Val = elastic_tensor[1,0,0,1]
    C_names = ['C11', 'C12', 'C13', 'C33', 'C44']
    C_values = [C11Val, C12Val, C13Val, C33Val, C44Val]

# Calculate the Young's modulus
pr = PoissonRatioVal[0]
ElasticModulusVal = 3 * BulkModulusVRHVal * (1 - 2 * pr) #Elastic moduli for homogeneous isotropic materials

# Create the DataFrame
dfCVal = pd.DataFrame({
    'Elastic constant': C_names,
    'Elastic constant value': C_values,
})

print(dfCVal)

  Elastic constant  Elastic constant value
0              C11                   274.0
1              C12                   175.0
2              C44                    89.0


## Calculations of (universal) elastic anisoptropy index
http://dx.doi.org/10.1103/PhysRevLett.101.055504

In [196]:
# Zener anisotropy index
ZenerAnisotropyIndex = 2*C44Val / (C11Val - C12Val)
ChungBuessemAnisotropyIndex = (ShearModulusVoigtVal - ShearModulusReussVal) / (ShearModulusVoigtVal + ShearModulusReussVal)
UniversalAnisotropyIndex = 10*(ChungBuessemAnisotropyIndex/(1-ChungBuessemAnisotropyIndex))

## Creation of the 3D surface elastic tensor

In [197]:
# ELATE ???

## Export materials data from MaterialsProject to Granta MI database

In [198]:
import datetime
import GRANTA_MIScriptingToolkit as gdl
from GRANTA_MIScriptingToolkit import granta as mpy
import numpy as np

usernameVal = 'STK_user'
passwordVal = 'P455w07d!'

session = mpy.connect('https://nanomecommons.grantami.com/mi_servicelayer', user_name=usernameVal, password=passwordVal)

dbKey = "MI_Blank_v2021"
tableName = "Materials"
folderName = "Python Demo"
recordName = "Iron"

db = session.get_db(db_key=dbKey)
table = db.get_table(tableName)
table_folders = table.children
["Name: {}, Type: {}".format(child.name, child.type) for child in table_folders]

['Name: _Sample Material , Type: Record',
 'Name: Metals and Alloys, Type: Folder',
 'Name: Ceramics and Glasses, Type: Folder',
 'Name: Complex Alloys, Type: Folder',
 'Name: Patterned Surfaces and Coatings, Type: Folder',
 'Name: Polymers, Type: Folder',
 'Name: Python Demo, Type: Folder']

In [199]:
SXmaterials = [child for child in table_folders if child.short_name == folderName][0]
["Name: {}, Type: {}".format(child.name, child.type) for child in SXmaterials.children]
recs = table.get_records_from_path(SXmaterials, [])
recs

[<Record long name: Iron_alpha_0K>]

In [200]:
session = mpy.connect('https://nanomecommons.grantami.com/mi_servicelayer', user_name=usernameVal, password=passwordVal)

dbKey = "MI_Blank_v2021"
tableName = "Materials"
folderName = "Python Demo"
recordName = "Iron"

if QUERY == 'mp-13':
    phaseName = 'alpha'
    materialName = 'Iron_'+ phaseName
    recordName = materialName+'_0K'
else:
    phaseName = 'beta'
    materialName = 'Iron_'+ phaseName
    recordName = materialName+'_0K'

db = session.get_db(db_key=dbKey)
table = db.get_table(tableName)
record=[]

try:
    #record = table.path_from(None, tree_path=['', folderName], end_node=recordName)
    record = table.search_for_records_by_name(recordName)
    record = record[0]
    print(record)
except:
    record = table.create_record(recordName, parent=SXmaterials, subsets={'Materials'}, color=mpy.RecordColor.White)

#Dataframe for phases
phaseName = pd.DataFrame({'Phase name': [phaseName], 'Volume fraction': [100]})

#Get the attributes
Source_value = record.attributes['Additional notes']
materialName_value = record.attributes['Grade name']
chemicalComposition_value = record.attributes['Composition summary']
density_value = record.attributes['Density']

crystalSystem_value = record.attributes['Crystal structure']
lattParam_a_value = record.attributes['Lattice constant (a parameter)']
lattParam_c_value = record.attributes['Lattice constant (c parameter)']
E_value = record.attributes["Young's modulus"]
pr_value = record.attributes["Poisson's ratio"]

# Results
Source_value.value = SourceRef + '_' + QUERY + '_' + str(TRef) + 'K'
materialName_value.value = materialName
chemicalComposition_value.value = chemical_info
density_value.value = {'low': DensityVal[0], 'high': DensityVal[0]}
crystalSystem_value.value = CSVal
lattParam_a_value.value = LattParamAVal[0]/10 #in nm
lattParam_c_value.value = LattParamCVal[0]/10 #in nm
E_value.value = {'low': ElasticModulusVal, 'high': ElasticModulusVal}
pr_value.value = {'low': PoissonRatioVal[0], 'high': PoissonRatioVal[0]}

# Missing crystalStructure_value, 
record.set_attributes([Source_value, \
                       materialName_value, chemicalComposition_value, \
                       density_value, \
                       crystalSystem_value, lattParam_a_value, lattParam_c_value, \
                       E_value, pr_value ])
record = session.update([record])

<Record long name: Iron_alpha_0K>
